# 07 — Final Results Aggregation (`07_results_table.ipynb`)

**Purpose:** Aggregate all `results/*.json` outputs across **Run A** (Clean ZWJ, $N=5{,}723$) and **Run B** (Emoji + Conversational Punctuation Kept, Gold-Safe Dedup, $N=5{,}834$) on the 300-row human gold benchmark into:
1. `results_table.csv` (and per-run CSVs `results_table_runA.csv`, `results_table_runB.csv`)
2. `results_table.tex` (publication-ready LaTeX `tabular` environment with best per column bolded via `\textbf{...}`).

## 1. Environment & Single Config Cell

In [ ]:
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import glob
import json
import numpy as np
import pandas as pd

from eval_utils import log_environment

# ── Single Config Cell ────────────────────────────────────────────────────────
RESULTS_DIR  = "results"
OUT_CSV      = "results_table.csv"
OUT_TEX      = "results_table.tex"

env_info = log_environment()

## 2. Load All `results/*.json` & Format Mean ± Std Table (Run A & Run B)

In [ ]:
ROW_SPEC = [
    ("Majority class",                ["01_majority_baseline"]),
    ("TF-IDF + LR (char)",            ["02_tfidf_lr_baseline_char", "02_tfidf_lr_baseline"]),
    ("TF-IDF + LR (word)",            ["02_tfidf_lr_baseline_word"]),
    ("XLM-R plain (noMLM+mean)",      ["03_xlmr_plain_baseline", "ablation_noMLM+mean"]),
    ("noMLM+attn",                    ["ablation_noMLM+attn"]),
    ("MLM+mean",                      ["ablation_MLM+mean"]),
    ("MLM+attn (proposed)",           ["ablation_MLM+attn"]),
    ("Single-task sentiment",         ["05_single_task_sentiment"]),
    ("Matched-compute control",       ["04_matched_compute_mlm_control", "04_matched_compute_mlm_control_genericMLM+attn"]),
]

METRIC_COLS = [
    ("sent_acc",     "Sent Acc"),
    ("sent_f1_3cls", "Sent F1 (3-cls)"),
    ("sent_f1_4cls", "Sent F1 (4-cls)"),
    ("intent_acc",   "Intent Acc"),
    ("intent_f1",    "Intent F1"),
    ("joint_acc",    "Joint Acc"),
]

json_records = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json"))):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "notebook" in data and "run" in data:
        json_records.append(data)

print(f"Loaded {len(json_records)} result JSON files from `{RESULTS_DIR}/`.")

def build_run_table(run_name: str):
    rows_raw = []
    for display_label, nb_tags in ROW_SPEC:
        matches = [r for r in json_records if r["run"] == run_name and r["notebook"] in nb_tags]
        by_seed = {}
        for tag in nb_tags:
            for r in matches:
                if r["notebook"] == tag and r["seed"] not in by_seed:
                    by_seed[r["seed"]] = r
        seed_list = [by_seed[k] for k in sorted(by_seed.keys(), key=str)]

        row_dict = {"Run": run_name, "Model": display_label, "n_seeds": len(seed_list)}
        for m_key, col_name in METRIC_COLS:
            vals = [x[m_key] for x in seed_list if x.get(m_key) is not None]
            if not vals:
                row_dict[f"{m_key}_mean"] = np.nan
                row_dict[f"{m_key}_std"]  = np.nan
                row_dict[col_name]        = "—"
            elif len(vals) == 1 and not seed_list[0].get("is_preaggregated", False):
                m = float(vals[0])
                row_dict[f"{m_key}_mean"] = m
                row_dict[f"{m_key}_std"]  = 0.0
                row_dict[col_name]        = f"{m:.3f}"
            else:
                m = float(np.mean(vals))
                s = float(seed_list[0].get(f"{m_key}_std", np.std(vals, ddof=0))) if len(vals) == 1 else float(np.std(vals, ddof=0))
                row_dict[f"{m_key}_mean"] = m
                row_dict[f"{m_key}_std"]  = s
                row_dict[col_name]        = f"{m:.3f} ± {s:.3f}" if s > 0 else f"{m:.3f}"
        rows_raw.append(row_dict)

    df_run = pd.DataFrame(rows_raw)

    best_indices = {}
    for m_key, col_name in METRIC_COLS:
        valid = df_run[f"{m_key}_mean"].dropna()
        if len(valid) > 0:
            max_val = valid.max()
            best_indices[m_key] = set(df_run.index[np.isclose(df_run[f"{m_key}_mean"], max_val, atol=1e-4)])
        else:
            best_indices[m_key] = set()

    return df_run, best_indices

df_runA, best_A = build_run_table("runA")
df_runB, best_B = build_run_table("runB")

display_cols = ["Run", "Model"] + [c for _, c in METRIC_COLS]
print("=== Run A (Clean ZWJ, N=5,723) — 300 Gold Test Rows ===")
display(df_runA[display_cols])

print("\n=== Run B (Emoji + Punctuation Kept, Gold-Safe Dedup, N=5,834) — 300 Gold Test Rows ===")
display(df_runB[display_cols])

## 3. Export `results_table.csv` and LaTeX `results_table.tex` (with Bold Best Per Column)

In [ ]:
combined_df = pd.concat([df_runA[display_cols], df_runB[display_cols]], ignore_index=True)
combined_df.to_csv(OUT_CSV, index=False)
df_runA[display_cols].to_csv("results_table_runA.csv", index=False)
df_runB[display_cols].to_csv("results_table_runB.csv", index=False)
print(f"Exported {OUT_CSV}, results_table_runA.csv, and results_table_runB.csv")

def to_latex_tabular(df_A, best_A, df_B, best_B) -> str:
    lines = [
        "% Auto-generated by 07_results_table.ipynb",
        "\begin{table*}[t]",
        "\caption{Baseline, factorial ablation, and control experiments evaluated on the 300-row human-verified gold benchmark (mean $\pm$ std over 3 seeds: 42, 43, 44). Best value in each column per preprocessing regime is shown in \textbf{bold}.}",
        "\label{tab:full_baseline_ablation_results}",
        "\centering",
        "\small",
        "\begin{tabular}{llcccccc}",
        "\toprule",
        "\textbf{Regime} & \textbf{Model} & \textbf{Sent Acc} & \textbf{Sent F1 (3-cls)} & \textbf{Sent F1 (4-cls)} & \textbf{Intent Acc} & \textbf{Intent F1} & \textbf{Joint Acc} \\",
        "\midrule",
    ]

    for regime_label, df_r, best_r in [
        ("Run A (Clean ZWJ)", df_A, best_A),
        ("Run B (Emoji+Punct)", df_B, best_B),
    ]:
        for idx, row in df_r.iterrows():
            r_col = regime_label if idx == 0 else ""
            m_col = row["Model"].replace("_", "\_")
            cells = []
            for m_key, col_name in METRIC_COLS:
                val_str = str(row[col_name]).replace("±", "$\pm$")
                if idx in best_r.get(m_key, set()) and val_str != "—":
                    val_str = f"\textbf{{{val_str}}}"
                cells.append(val_str)
            lines.append(f"{r_col} & {m_col} & " + " & ".join(cells) + " \\")
        if regime_label.startswith("Run A"):
            lines.append("\midrule")

    lines.extend([
        "\bottomrule",
        "\end{tabular}",
        "\end{table*}",
    ])
    return "\n".join(lines)

latex_str = to_latex_tabular(df_runA, best_A, df_runB, best_B)
with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex_str + "\n")

print(f"Exported {OUT_TEX}:\n")
print(latex_str)